# Data Prep

In [ ]:
import zipfile
from pathlib import Path
import rasterio
from rasterio.merge import merge
import shutil


## Merging zipfiled tifs

In [ ]:

country = "dnk"
year = 2018
dataset = "tree_cover_density"
folder = "Tree_Density"

# Use Linux paths when running Python in WSL
input_dir = f"/home/georg/data/LEON_P5_BII/EO_data_raw/{folder}/{dataset}_{country}_{year}"
output_dir = f"/home/georg/data/LEON_P5_BII/EO_data_prep/{folder}"

# Check if input directory exists
input_path = Path(input_dir)
print(f"Input directory: {input_path}")
print(f"Directory exists: {input_path.exists()}")

# Get all zip files
zip_files = list(input_path.glob("*.zip"))
print(f"Found {len(zip_files)} zip files")

if not zip_files:
    print("ERROR: No zip files found!")
    raise FileNotFoundError(f"No zip files found in {input_dir}")

# Create temp directory
Path("temp").mkdir(exist_ok=True)

# Extract and merge tif files
tif_files = []
for zf in zip_files:
    print(f"Processing: {zf.name}")
    with zipfile.ZipFile(zf) as z:
        # Find .tif file in zip
        tif_matches = [n for n in z.namelist() if n.endswith('.tif')]
        if not tif_matches:
            print(f"  WARNING: No .tif files in {zf.name}")
            continue
        tif_name = tif_matches[0]
        z.extract(tif_name, "temp")
        tif_files.append(f"temp/{tif_name}")

print(f"\nTotal .tif files extracted: {len(tif_files)}")

# Open all tif files
src_files = [rasterio.open(f) for f in tif_files]

# Merge them
mosaic, out_transform = merge(src_files)

# Save merged result
Path(output_dir).mkdir(parents=True, exist_ok=True)
with rasterio.open(
    f"{output_dir}/{dataset}_{country}_{year}_merged.tif", "w",
    driver="GTiff",
    height=mosaic.shape[1],
    width=mosaic.shape[2],
    count=mosaic.shape[0],
    dtype=mosaic.dtype,
    transform=out_transform,
    crs=src_files[0].crs,
    compress='LZW',           # LZW compression
    # predictor=2,              # Improves compression for continuous data not categorical (assume similar neighbors)
    tiled=True,               # Better performance with compression
    blockxsize=256,           # Tile size
    blockysize=256
) as dest:
    dest.write(mosaic)

print(f"Merged file saved to: {output_dir}/{dataset}_{country}_{year}_merged.tif")

# Cleanup
for src in src_files:
    src.close()

shutil.rmtree("temp")

Input directory: /home/georg/data/LEON_P5_BII/EO_data_raw/Tree_Density/tree_cover_density_dnk_2018
Directory exists: True
Found 23 zip files
Processing: CLMS_HRLVLCC_TCD_S2018_R10m_E44N37_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_TCD_S2018_R10m_E42N35_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_TCD_S2018_R10m_E44N36_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_TCD_S2018_R10m_E42N34_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_TCD_S2018_R10m_E43N35_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_TCD_S2018_R10m_E44N38_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_TCD_S2018_R10m_E41N34_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_TCD_S2018_R10m_E46N37_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_TCD_S2018_R10m_E46N38_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_TCD_S2018_R10m_E43N37_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_TCD_S2018_R10m_E45N38_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_TCD_S2018_R10m_E45N36_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_TCD_S2018_R10m_E45N37_03035_V01_R00.zip
Processing: CL

In [ ]:
# Check directory
ghsl_base = Path("/home/georg/data/LEON_P5_BII/EO_data_raw/GHSL")

print(f"GHSL directory exists: {ghsl_base.exists()}")
print(f"\nContents of {ghsl_base}:")
print("="*60)

if ghsl_base.exists():
    # Show subdirectories and their contents
    for item in sorted(ghsl_base.iterdir()):
        if item.is_dir():
            print(f"\n📁 {item.name}/")
            # Show what's inside each subdirectory (one level deep)
            for subitem in sorted(item.iterdir())[:10]:  # Limit to first 10 items
                if subitem.is_dir():
                    print(f"   📁 {subitem.name}/")
                else:
                    print(f"   📄 {subitem.name}")
            if len(list(item.iterdir())) > 10:
                print(f"   ... and {len(list(item.iterdir())) - 10} more items")
        else:
            print(f"📄 {item.name}")
else:
    print("Directory does not exist!")

GHSL directory exists: True

Contents of /home/georg/data/LEON_P5_BII/EO_data_raw/GHSL:

📁 2015/
   📁 GHS_POP_E2015_GLOBE_R2023A_54009_100_V1_0_R3_C18/
   📁 GHS_POP_E2015_GLOBE_R2023A_54009_100_V1_0_R3_C19/
   📁 GHS_POP_E2015_GLOBE_R2023A_54009_100_V1_0_R4_C18/
   📁 GHS_POP_E2015_GLOBE_R2023A_54009_100_V1_0_R4_C19/

📁 2020/
   📁 GHS_POP_E2020_GLOBE_R2023A_54009_100_V1_0_R3_C18/
   📁 GHS_POP_E2020_GLOBE_R2023A_54009_100_V1_0_R3_C19/
   📁 GHS_POP_E2020_GLOBE_R2023A_54009_100_V1_0_R4_C18/
   📁 GHS_POP_E2020_GLOBE_R2023A_54009_100_V1_0_R4_C19/

📁 2025/
   📁 GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0_R3_C18/
   📁 GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0_R3_C19/
   📁 GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0_R4_C18/
   📁 GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0_R4_C19/


In [3]:
# Merging multiple countries and years in folder (not zipped)

import rasterio
from rasterio.merge import merge
from pathlib import Path

dataset = "ghsl_pop"
output_base = "/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL"

# Define what to process
countries_years = {
    "ndl": [2015, 2020, 2025],
    "cnk": [2015, 2020, 2025],
    "uk": [2015, 2025]
}


for country, years in countries_years.items():
    for year in years:
        print(f"\n{'='*60}")
        print(f"Processing: {country.upper()} - {year}")
        print(f"{'='*60}")
        
        # Adjust base directory structure based on your actual folder organization
        base_dir = f"/home/georg/data/LEON_P5_BII/EO_data_raw/GHSL/{year}"
        
        input_path = Path(base_dir)
        if not input_path.exists():
            print(f"⚠ Directory not found: {base_dir}")
            continue
        
        # Find all .tif files
        tif_files = list(input_path.rglob("*.tif"))
        print(f"Found {len(tif_files)} .tif files")
        
        if not tif_files:
            print(f"⚠ No .tif files found for {country} {year}")
            continue
        
        # Open, merge, and save
        try:
            src_files = [rasterio.open(str(f)) for f in tif_files]
            mosaic, out_transform = merge(src_files)
            
            output_dir = Path(output_base)
            output_dir.mkdir(parents=True, exist_ok=True)
            output_file = output_dir / f"{dataset}_{country}_{year}_merged.tif"
            
            with rasterio.open(
                output_file, "w",
                driver="GTiff",
                height=mosaic.shape[1],
                width=mosaic.shape[2],
                count=mosaic.shape[0],
                dtype=mosaic.dtype,
                transform=out_transform,
                crs=src_files[0].crs,
                compress='LZW',
                tiled=True,
                blockxsize=256,
                blockysize=256
            ) as dest:
                dest.write(mosaic)
            
            # Cleanup
            for src in src_files:
                src.close()
            
            print(f"✓ Saved: {output_file}")
            
        except Exception as e:
            print(f"✗ Error processing {country} {year}: {e}")


Processing: NDL - 2015
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_ndl_2015_merged.tif

Processing: NDL - 2020
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_ndl_2020_merged.tif

Processing: NDL - 2025
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_ndl_2025_merged.tif

Processing: CNK - 2015
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_cnk_2015_merged.tif

Processing: CNK - 2020
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_cnk_2020_merged.tif

Processing: CNK - 2025
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_cnk_2025_merged.tif

Processing: UK - 2015
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_uk_2015_merged.tif

Processing: UK - 2025
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop

In [ ]:
dataset = "ghsl_pop"
output_base = "/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL"

# Clip to Country boundaries

In [2]:
import geopandas as gpd
from pathlib import Path

# Directory with gpkg files
gpkg_dir = Path("/home/georg/data/LEON_P5_BII/Country_Boundaries")

# Load all gpkg files
uk = gpd.read_file(gpkg_dir / "gadm41_GBR_adm2_filtered4_3035.gpkg")
dnk = gpd.read_file(gpkg_dir / "gadm41_dnk_adm_0_3035.gpkg")
nld = gpd.read_file(gpkg_dir / "gadm41_nld_adm_0_3035.gpkg")

In [ ]:
from rasterio.mask import mask
from pathlib import Path

### For folder ###

## Config ##
aoi = uk # Using country boundaries for clipping

country = "nld"
year = 2023
dataset = "bare_before"
folder = "Bare"


# Find all .tif files in subfolders
tif_dir = Path(f"/home/georg/data/LEON_P5_BII/EO_data_prep/{folder}/{dataset}_{country}_{year}_merged.tif")
tif_files = list(tif_dir.rglob("*.tif"))

print(f"Found {len(tif_files)} .tif files")

# Load and clip each tif with each country boundary
for tif_file in tif_files:
    print(f"\nProcessing: {tif_file.name}")
    
    with rasterio.open(tif_file) as src:
        # Clip with GBR
        aoi_geom = [aoi.geometry.unary_union]
        aoi_clipped, aoi_transform = mask(src, aoi_geom, crop=True)
        
        # Save clipped result
        output_path = f"/home/georg/data/LEON_P5_BII/EO_data_prep/{folder}/{dataset}_{country}_{year}.tif"
        with rasterio.open(output_path, 'w',
                          driver='GTiff',
                          height=aoi_clipped.shape[1],
                          width=aoi_clipped.shape[2],
                          count=src.count,
                          dtype=aoi_clipped.dtype,
                          crs=src.crs,
                          transform=aoi_transform,
                          compress='LZW') as dst:
            dst.write(aoi_clipped)
        
        print(f"Saved: {output_path}")

Found 9 .tif files

Processing: canopy_height_netherlands_2m-0000000000-0000000000.tif


NameError: name 'rasterio' is not defined

In [14]:
import rasterio
from rasterio.mask import mask
from pathlib import Path

### For files ###

## Config ##
aoi = nld # Using country boundaries for clipping

country = "nld"
year = 2023
dataset = "bare_after"
folder = "Bare" 

# Path can be a file or folder
tif_path = Path(f"/home/georg/data/LEON_P5_BII/EO_data_prep/{folder}/{dataset}_{country}_{year}.tif")

# Check if it's a file or folder
if tif_path.is_file():
    tif_files = [tif_path]
elif tif_path.is_dir():
    tif_files = list(tif_path.rglob("*.tif"))
else:
    raise FileNotFoundError(f"Path not found: {tif_path}")

print(f"Found {len(tif_files)} .tif file(s)")

# Load and clip each tif
for tif_file in tif_files:
    print(f"\nProcessing: {tif_file.name}")
    
    with rasterio.open(tif_file) as src:
        # Clip with AOI
        aoi_geom = [aoi.geometry.unary_union]
        aoi_clipped, aoi_transform = mask(src, aoi_geom, crop=True)
        
        # Generate output filename
        output_name = tif_file.stem + "_clip.tif"
        output_path = tif_file.parent / output_name
        
        # Save clipped result
        with rasterio.open(output_path, 'w',
                          driver='GTiff',
                          height=aoi_clipped.shape[1],
                          width=aoi_clipped.shape[2],
                          count=src.count,
                          dtype=aoi_clipped.dtype,
                          crs=src.crs,
                          transform=aoi_transform,
                          compress='LZW') as dst:
            dst.write(aoi_clipped)
        
        print(f"Saved: {output_path}")

Found 1 .tif file(s)

Processing: bare_after_nld_2023.tif
Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/Bare/bare_after_nld_2023_clip.tif


## Mask based on Raster

In [4]:
# Batch compute

import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
from pathlib import Path

# Config
nitrogen_path = Path("/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_2015_merged.tif")
lc_dir = Path("/home/georg/data/LEON_P5_BII/EO_data_prep/CLCplus")
output_dir = Path("/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL")

output_dir.mkdir(exist_ok=True, parents=True)

# Process each country
for lc_path in lc_dir.glob("clcplus_*_2023.tif"):
    country = lc_path.stem.split('_')[1]  # dnk, nld, uk
    output_path = output_dir / f"ghsl_{country}_15.tif"
    
    print(f"Processing {country.upper()}...")
    
    with rasterio.open(lc_path) as lc, rasterio.open(nitrogen_path) as src:
        # Get valid LC pixel mask
        lc_data = lc.read(1)
        valid_mask = lc_data != lc.nodata if lc.nodata else np.ones_like(lc_data, dtype=bool)
        
        # Reproject nitrogen to LC grid
        output = np.empty((src.count, lc.height, lc.width), dtype=src.dtypes[0])
        
        reproject(
            source=rasterio.band(src, range(1, src.count + 1)),
            destination=output,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=lc.transform,
            dst_crs=lc.crs,
            resampling=Resampling.bilinear
        )
        
        # Mask invalid LC pixels
        output[:, ~valid_mask] = src.nodata if src.nodata else -9999
        
        # Save
        profile = lc.profile.copy()
        profile.update(dtype=src.dtypes[0], count=src.count, nodata=src.nodata or -9999, compress='LZW')
        
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(output)
    
    print(f"  ✓ {output_path}")

print("\nDone!")

Processing NLD...
  ✓ /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_nld_15.tif
Processing UK...


: 

In [4]:
# Batch compute - preserve GHSL resolution, clip to LC extent

import rasterio
from rasterio.warp import reproject, calculate_default_transform, Resampling
from rasterio.mask import mask
from shapely.geometry import box
import numpy as np
from pathlib import Path


dataset = "ghsl_pop_2025_merged"
year = 2025

# Config
ghsl_path = Path(f"/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/{dataset}.tif")
lc_dir = Path("/home/georg/data/LEON_P5_BII/EO_data_prep/CLCplus")
output_dir = Path("/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL")

output_dir.mkdir(exist_ok=True, parents=True)

# Process each country
for lc_path in lc_dir.glob("clcplus_*_2023.tif"):
    country = lc_path.stem.split('_')[1]  # dnk, nld, uk
    output_path = output_dir / f"ghsl_{country}_{year}.tif"
    
    print(f"Processing {country.upper()}...")
    
    with rasterio.open(lc_path) as lc, rasterio.open(ghsl_path) as ghsl:
        
        # Get LC bounds in its CRS
        lc_bounds = lc.bounds
        
        # If CRS differs, reproject LC bounds to GHSL CRS
        if lc.crs != ghsl.crs:
            from rasterio.warp import transform_bounds
            ghsl_bounds = transform_bounds(lc.crs, ghsl.crs, *lc_bounds)
        else:
            ghsl_bounds = lc_bounds
        
        # Create bounding box geometry
        bbox = box(*ghsl_bounds)
        
        # Clip GHSL to LC extent (keeps GHSL resolution)
        clipped_data, clipped_transform = mask(
            ghsl, 
            [bbox], 
            crop=True,
            all_touched=True  # Include pixels that touch the boundary
        )
        
        # If CRS differs, reproject the clipped data to LC CRS
        if lc.crs != ghsl.crs:
            print(f"  Reprojecting from {ghsl.crs} to {lc.crs}...")
            
            # Calculate transform for reprojected data (keeps GHSL resolution)
            dst_transform, dst_width, dst_height = calculate_default_transform(
                ghsl.crs, 
                lc.crs, 
                clipped_data.shape[2], 
                clipped_data.shape[1],
                *ghsl_bounds,
                resolution=ghsl.res  # Keep original resolution
            )
            
            # Reproject
            reprojected = np.empty((ghsl.count, dst_height, dst_width), dtype=ghsl.dtypes[0])
            reproject(
                source=clipped_data,
                destination=reprojected,
                src_transform=clipped_transform,
                src_crs=ghsl.crs,
                dst_transform=dst_transform,
                dst_crs=lc.crs,
                resampling=Resampling.bilinear
            )
            
            output_data = reprojected
            output_transform = dst_transform
            output_crs = lc.crs
        else:
            output_data = clipped_data
            output_transform = clipped_transform
            output_crs = ghsl.crs
        
        # Save with GHSL properties but aligned to LC extent
        profile = ghsl.profile.copy()
        profile.update(
            height=output_data.shape[1],
            width=output_data.shape[2],
            transform=output_transform,
            crs=output_crs,
            compress='LZW',
            tiled=True
        )
        
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(output_data)
    
    print(f"  ✓ Resolution: {ghsl.res[0]:.2f}m")
    print(f"  ✓ Shape: {output_data.shape}")
    print(f"  ✓ {output_path}")

print("\nDone!")

Processing NLD...
  Reprojecting from ESRI:54009 to EPSG:3035...
  ✓ Resolution: 100.00m
  ✓ Shape: (1, 3489, 3457)
  ✓ /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_nld_2025.tif
Processing UK...
  Reprojecting from ESRI:54009 to EPSG:3035...
  ✓ Resolution: 100.00m
  ✓ Shape: (1, 4378, 5448)
  ✓ /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_uk_2025.tif
Processing DNK...
  Reprojecting from ESRI:54009 to EPSG:3035...
  ✓ Resolution: 100.00m
  ✓ Shape: (1, 3799, 5121)
  ✓ /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_dnk_2025.tif

Done!


In [ ]:
# Single file compute

import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np

# Paths
nitrogen_path = "/home/georg/data/LEON_P5_BII/EO_data_raw/Nitrogen/nitrogen_soilgrids.tif"
lc_path = "/home/georg/data/LEON_P5_BII/EO_data_prep/CLCplus/clcplus_nld_2023.tif"
output_path = "/home/georg/data/LEON_P5_BII/EO_data_prep/Nitrogen/nitrogen_nld_2023_clipped.tif"

with rasterio.open(lc_path) as lc, rasterio.open(nitrogen_path) as src:
    # Read LC data to get valid pixel mask
    lc_data = lc.read(1)
    valid_mask = lc_data != lc.nodata if lc.nodata else np.ones_like(lc_data, dtype=bool)
    
    # Create output array matching LC dimensions
    output = np.empty((src.count, lc.height, lc.width), dtype=src.dtypes[0])
    
    # Reproject nitrogen to LC grid
    reproject(
        source=rasterio.band(src, range(1, src.count + 1)),
        destination=output,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=lc.transform,
        dst_crs=lc.crs,
        resampling=Resampling.bilinear
    )
    
    # Mask invalid LC pixels
    output[:, ~valid_mask] = src.nodata if src.nodata else -9999
    
    # Save
    profile = lc.profile.copy()
    profile.update(dtype=src.dtypes[0], count=src.count, nodata=src.nodata or -9999, compress='LZW')
    
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(output)

print(f"✓ Saved: {output_path}")